# 5. 工具调用流程分析
LangChain 的 Agent 会将 模型 与 工具 结合起来，在实现上由一个基于 LangGraph 的图结构来编排执
行流程，如下所示。

这与前文得到的 Agent 图结构是一致的，本质上就是经典的 ReAct 结构：一个具备“ 思考-行动-观察 ”不断循环的自主工作者。

> 用户问题 → AI 思考 → 调用工具 → 观察结果 → 继续思考 → ... → 最终答案

当用户提出一个复杂需求时，Agent会像人类一样，先理解任务、规划步骤、使用合适的工具（如搜索网络、查询数据库、执行计算）获取信息，Agent 会在一个循环中 反复调用模型和工具 ，直到某次模型输出中 不再包含工具调用 则结束，最后综合所有信息给出最终答案。

## 5.1 重试机制

Agent可以在工具调用结果不满足要求时，自主重试。

In [3]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from rich import print as rprint
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.tools import BaseTool
from langchain.messages import SystemMessage, HumanMessage

load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致
flag = 0  # 用于模拟天气服务的临时故障
@tool(parse_docstring=True)
def get_weather(city: str):
    """
    天气查询工具
    
    Args:
        city: 城市名称
    """
    global flag
    flag += 1
    if flag < 3:
    # raise Exception("暂时无法访问")
        return "TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试"
    return f"{city}今天天气挺好"


model= ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

agent = create_agent(
    model=model, tools=[get_weather],
    system_prompt="""
    你是一个天气助手。
    当工具返回以 'TEMP_UNAVAILABLE:' 开头的结果时，
    说明是临时故障，不要立即放弃；
    你应再次调用同一个工具，最多重试 3 次。
    如果 3 次后仍失败，再向用户说明服务暂时不可用。
    """
)  # 传入的是 ChatOpenAI 实例
messages = [
    HumanMessage("你好，杭州今天的天气如何？")
]

response = agent.invoke({"messages": messages})
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，杭州今天的天气如何？
================================== Ai Message ==================================

[{'arguments': '{"city":"杭州"}', 'call_id': 'call_oJSlD5j9tGjM5Cr2gHV4rm3d', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_0137c9b9efa4dcc8016aba07eb831887d29b4e3a65ba55faa7', 'status': 'completed', 'internal_chat_message_metadata_passthrough': {'create_time': 1790576618.923104, 'turn_id': '01a0e6ae-eb93-7994-a91f-95b85d319924'}, 'metadata': {'turn_id': '01a0e6ae-eb93-7994-a91f-95b85d319924'}}]
Tool Calls:
  get_weather (call_oJSlD5j9tGjM5Cr2gHV4rm3d)
 Call ID: call_oJSlD5j9tGjM5Cr2gHV4rm3d
  Args:
    city: 杭州
================================= Tool Message =================================
Name: get_weather

TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试
================================== Ai Message ==================================

[{'arguments': '{"city":"杭州"}', 'call_id': 'call_JW2pCENLO7hrGbZMAqN62IuU', 'n

In [4]:
rprint(response)

{
    'messages': [
        HumanMessage(
            content='你好，杭州今天的天气如何？',
            additional_kwargs={},
            response_metadata={},
            id='826e978f-4553-4510-bba6-617185a6a4cb'
        ),
        AIMessage(
            content=[
                {
                    'arguments': '{"city":"杭州"}',
                    'call_id': 'call_oJSlD5j9tGjM5Cr2gHV4rm3d',
                    'name': 'get_weather',
                    'type': 'function_call',
                    'id': 'fc_0137c9b9efa4dcc8016aba07eb831887d29b4e3a65ba55faa7',
                    'status': 'completed',
                    'internal_chat_message_metadata_passthrough': {
                        'create_time': 1790576618.923104,
                        'turn_id': '01a0e6ae-eb93-7994-a91f-95b85d319924'
                    },
                    'metadata': {'turn_id': '01a0e6ae-eb93-7994-a91f-95b85d319924'}
                }
            ],
            additional_kwargs={},
            response_metadata={
                'id': 'resp_0137c9b9efa4dcc8016aba07eab94487d2bf4bed32df19216c',
                'created_at': 1790576618.0,
                'metadata': {},
                'model': 'gpt-6-luna',
                'object': 'response',
                'service_tier': 'default',
                'status': 'completed',
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna'
            },
            id='resp_0137c9b9efa4dcc8016aba07eab94487d2bf4bed32df19216c',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '杭州'},
                    'id': 'call_oJSlD5j9tGjM5Cr2gHV4rm3d',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 4511,
                'output_tokens': 18,
                'total_tokens': 4529,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试',
            name='get_weather',
            id='46201f5f-80df-434b-872f-09508e17bbbd',
            tool_call_id='call_oJSlD5j9tGjM5Cr2gHV4rm3d'
        ),
        AIMessage(
            content=[
                {
                    'arguments': '{"city":"杭州"}',
                    'call_id': 'call_JW2pCENLO7hrGbZMAqN62IuU',
                    'name': 'get_weather',
                    'type': 'function_call',
                    'id': 'fc_0137c9b9efa4dcc8016aba07f58db487d2a6337c639846aa4d',
                    'status': 'completed',
                    'internal_chat_message_metadata_passthrough': {
                        'create_time': 1790576629.068997,
                        'turn_id': '01a0e6af-1228-7e7c-9d0c-7e2a089b6248'
                    },
                    'metadata': {'turn_id': '01a0e6af-1228-7e7c-9d0c-7e2a089b6248'}
                }
            ],
            additional_kwargs={},
            response_metadata={
                'id': 'resp_0137c9b9efa4dcc8016aba07f4cc6c87d282478249a40d4903',
                'created_at': 1790576628.0,
                'metadata': {},
                'model': 'gpt-6-luna',
                'object': 'response',
                'service_tier': 'default',
                'status': 'completed',
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna'
            },
            id='resp_0137c9b9efa4dcc8016aba07f4cc6c87d282478249a40d4903',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '杭州'},
                    'id': 'call_JW2pCENLO7hrGbZMAqN62IuU',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 4556,
                'outp